In [1]:
import pandas as pd

df = pd.read_excel("ENERGY EFFICIENCY & WASTE.xlsx")
df.head()

,Timestamp,MachineID,Plant,Temperature,Vibration,Pressure,EnergyConsumption,ProductionUnits,DefectCount,MaintenanceFlag
0,2025-01-01 00:00:00,140,Plant_C,73.585558,2.266289,20.785937,289.823429,128,2,0
1,2025-01-01 01:00:00,115,Plant_A,76.981562,3.326905,31.464005,181.688138,160,2,0
2,2025-01-01 02:00:00,145,Plant_C,68.554217,5.285392,30.768659,257.077101,79,3,0
3,2025-01-01 03:00:00,108,Plant_C,92.756558,6.289777,33.924396,217.046652,146,5,1
4,2025-01-01 04:00:00,122,Plant_C,76.174981,7.844886,25.129085,251.312522,142,4,0


In [2]:
df.columns

Index(['Timestamp', 'MachineID', 'Plant', 'Temperature', 'Vibration',
       'Pressure', 'EnergyConsumption', 'ProductionUnits', 'DefectCount',
       'MaintenanceFlag'],
      dtype='object')

In [3]:
import numpy as np

df['EnergyPerUnit'] = np.where(
    df['ProductionUnits'] == 0,
    0,
    df['EnergyConsumption'] / df['ProductionUnits']
)

df.head()

,Timestamp,MachineID,Plant,Temperature,Vibration,Pressure,EnergyConsumption,ProductionUnits,DefectCount,MaintenanceFlag,EnergyPerUnit
0,2025-01-01 00:00:00,140,Plant_C,73.585558,2.266289,20.785937,289.823429,128,2,0,2.264246
1,2025-01-01 01:00:00,115,Plant_A,76.981562,3.326905,31.464005,181.688138,160,2,0,1.135551
2,2025-01-01 02:00:00,145,Plant_C,68.554217,5.285392,30.768659,257.077101,79,3,0,3.254141
3,2025-01-01 03:00:00,108,Plant_C,92.756558,6.289777,33.924396,217.046652,146,5,1,1.486621
4,2025-01-01 04:00:00,122,Plant_C,76.174981,7.844886,25.129085,251.312522,142,4,0,1.769806


In [4]:
from scipy.stats import zscore

df['Z_score'] = zscore(df['EnergyPerUnit'])

df.head()

,Timestamp,MachineID,Plant,Temperature,Vibration,Pressure,EnergyConsumption,ProductionUnits,DefectCount,MaintenanceFlag,EnergyPerUnit,Z_score
0,2025-01-01 00:00:00,140,Plant_C,73.585558,2.266289,20.785937,289.823429,128,2,0,2.264246,-0.050664
1,2025-01-01 01:00:00,115,Plant_A,76.981562,3.326905,31.464005,181.688138,160,2,0,1.135551,-1.127074
2,2025-01-01 02:00:00,145,Plant_C,68.554217,5.285392,30.768659,257.077101,79,3,0,3.254141,0.893377
3,2025-01-01 03:00:00,108,Plant_C,92.756558,6.289777,33.924396,217.046652,146,5,1,1.486621,-0.792267
4,2025-01-01 04:00:00,122,Plant_C,76.174981,7.844886,25.129085,251.312522,142,4,0,1.769806,-0.522199


In [5]:
import numpy as np

threshold = np.percentile(abs(df['Z_score']), 99)

anomalies = df[abs(df['Z_score']) >= threshold]

anomalies[['Timestamp', 'MachineID', 'Plant', 'EnergyPerUnit', 'Z_score']]

,Timestamp,MachineID,Plant,EnergyPerUnit,Z_score
94,2025-01-04 22:00:00,126,Plant_A,6.457084,3.947951
129,2025-01-06 09:00:00,118,Plant_A,5.658194,3.186068
218,2025-01-10 02:00:00,105,Plant_C,6.690292,4.170356
226,2025-01-10 10:00:00,121,Plant_B,5.951417,3.465708
679,2025-01-29 07:00:00,115,Plant_C,6.047888,3.557710
...,...,...,...,...,...
19747,2027-04-03 19:00:00,126,Plant_B,6.235666,3.736790
19811,2027-04-06 11:00:00,144,Plant_A,5.721768,3.246697
19891,2027-04-09 19:00:00,104,Plant_A,6.572819,4.058325
19933,2027-04-11 13:00:00,108,Plant_C,5.553653,3.086370


In [6]:
anomalies.to_excel("Energy_Spikes_Output.xlsx", index=False)